# Anti-Money Laundering (AML) Detection: LightGBM vs PyTorch GraphSAGE

This notebook demonstrates that a simple 40-line PyTorch Graph Neural Network (GNN) can outperform a highly engineered Tabular model (LightGBM) for detecting financial fraud (Anti-Money Laundering).

**Methodology:**
To eliminate temporal leakage (a common trap in AML benchmarks), we formulate this as a strict **Retrospective Node Classification** task. We take a snapshot of transactions over a chronological window. At exactly midnight on the final day, we classify all active nodes using only historical edges within that window.
* **Train Graph (Days 1-10):** We predict fraud for nodes active in this period based on their Day 1-10 topology.
* **Test Graph (Days 11-15):** We predict fraud for nodes active in this period based on their Day 11-15 topology.
Because we classify nodes *retrospectively* at the end of the window, no future edge data is used.

Furthermore, to ensure a fair comparison, LightGBM receives the same base features plus **NetworkX Weighted PageRank** and 1-hop neighbor aggregates. This guarantees LightGBM has access to global structural topology, preventing a "strawman" baseline.

* **LightGBM:** Relies heavily on hand-crafted aggregation features (in/out volumes, max amounts, neighbor averages, weighted PageRank).
* **GraphSAGE (Pure PyTorch):** Uses the same base features, but leverages the *graph topology* via message passing, weighting edges by transaction amounts. It does not require complex C++ libraries like PyTorch Geometric.

Let's begin by generating the dataset.

In [ ]:
!pip install polars lightgbm scikit-learn networkx
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import numpy as np
from sklearn.metrics import precision_recall_curve, auc
import lightgbm as lgb
import matplotlib.pyplot as plt
import networkx as nx

## 1. Shared GNN Architecture (Pure PyTorch)
We define a highly efficient 40-line GraphSAGE model using PyTorch's built-in sparse matrix multiplication. No PyTorch Geometric required.

In [ ]:
class PureTorchGraphSAGE(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        self.lin_l = nn.Linear(in_channels, hidden_channels)
        self.lin_r = nn.Linear(in_channels, hidden_channels)
        self.lin_out = nn.Linear(hidden_channels, out_channels)

    def forward(self, x, edge_index, edge_weight):
        # We use edge_weight (normalized amount) in the adjacency matrix 
        # so message passing scales neighbor signals by financial volume.
        row, col = edge_index
        adj = torch.sparse_coo_tensor(
            torch.stack([col, row]), 
            edge_weight, 
            (x.size(0), x.size(0))
        )
        
        deg = torch.sparse.sum(adj, dim=1).to_dense().clamp(min=1e-5).unsqueeze(-1)
        aggr_out = torch.sparse.mm(adj, x) / deg
        
        out = self.lin_l(x) + self.lin_r(aggr_out)
        out = F.relu(out)
        return self.lin_out(out)

## 2. Load Data and Inductive Chronological Split
We download the generated synthetic AML dataset (10,000 agents, 15 days) and split it strictly by time to prevent temporal leakage.

In [ ]:
# For this Colab, we will download a script to generate the synthetic transactions locally.
!wget -q https://raw.githubusercontent.com/synthfin/synthfin-aml/main/generator.py -O generator.py

from generator import FraudGraphGenerator
print("Generating 15-day continuous data (10,000 agents)...")
gen = FraudGraphGenerator(seed=42)
gen.generate_transactions(agents=10000, days=15)
nodes_df, edges_df = gen.to_dataframes()

# Inductive Chronological Split
min_ts = edges_df['timestamp'].min()
max_ts = edges_df['timestamp'].max()
split_ts = min_ts + (max_ts - min_ts) * (10 / 15)

train_edges = edges_df[edges_df['timestamp'] <= split_ts].copy()
test_edges = edges_df[edges_df['timestamp'] > split_ts].copy()
print(f"Train edges: {len(train_edges)}, Test edges: {len(test_edges)}")

## 3. Shared Feature Engineering & Graph Extraction
Both LightGBM and the GNN receive the exact same 8 base features. We also give LightGBM 1-hop neighbor features and Weighted PageRank to ensure a fair fight.

In [ ]:
def extract_graph_data(edges, all_nodes_df):
    active_nodes = set(edges['source_id']).union(set(edges['target_id']))
    nodes = all_nodes_df[all_nodes_df['agent_id'].isin(active_nodes)].copy()
    
    out_degree = edges.groupby('source_id').size().rename('out_degree')
    in_degree = edges.groupby('target_id').size().rename('in_degree')
    out_volume = edges.groupby('source_id')['amount'].sum().rename('out_volume')
    in_volume = edges.groupby('target_id')['amount'].sum().rename('in_volume')
    out_max_amt = edges.groupby('source_id')['amount'].max().rename('out_max_amt')
    in_max_amt = edges.groupby('target_id')['amount'].max().rename('in_max_amt')

    # FAIRNESS: Neighbor Features for LightGBM
    edges_with_vol = edges.merge(in_volume, left_on='target_id', right_index=True, how='left')
    edges_with_vol = edges_with_vol.merge(out_volume, left_on='source_id', right_index=True, how='left')
    
    nbr_in_vol = edges_with_vol.groupby('source_id')['in_volume'].mean().rename('nbr_in_volume')
    nbr_out_vol = edges_with_vol.groupby('target_id')['out_volume'].mean().rename('nbr_out_volume')
    
    # FAIRNESS: Weighted PageRank for LightGBM
    # We use the 'edges' dataframe passed to this function, which is strictly bounded
    # by the time window (e.g. Days 1-10 for train, 11-15 for test). No future leakage!
    G = nx.from_pandas_edgelist(edges, 'source_id', 'target_id', edge_attr='amount', create_using=nx.DiGraph())
    pagerank = nx.pagerank(G, weight='amount')
    pr_series = pd.Series(pagerank, name='pagerank')

    features = nodes.set_index('agent_id').copy()
    features = features.join([out_degree, in_degree, out_volume, in_volume, out_max_amt, in_max_amt, nbr_in_vol, nbr_out_vol, pr_series]).fillna(0)
    
    y = features['is_fraud'].values
    
    # Normalization
    X_df = features.drop(columns=['profile', 'is_fraud'])
    for col in ['initial_balance', 'out_volume', 'in_volume', 'out_max_amt', 'in_max_amt', 'nbr_in_volume', 'nbr_out_volume', 'pagerank']:
        X_df[col] = np.log1p(X_df[col])
        
    X = X_df.values
    
    # Map edge_index for PyTorch
    agent2idx = {agent_id: i for i, agent_id in enumerate(features.index)}
    src = edges['source_id'].map(agent2idx).values
    dst = edges['target_id'].map(agent2idx).values
    edge_index = torch.tensor(np.vstack([src, dst]), dtype=torch.long)
    edge_weight = torch.tensor(np.log1p(edges['amount'].values), dtype=torch.float32)
    
    return X, y, edge_index, edge_weight

print('Building Train Graph (Days 1-10)...')
t0_feat = time.time()
X_train, y_train, edge_index_train, ew_train = extract_graph_data(train_edges, nodes_df)

print('Building Test Graph (Days 11-15)...')
X_test, y_test, edge_index_test, ew_test = extract_graph_data(test_edges, nodes_df)

# Standardize
scaler_mean = X_train.mean(axis=0)
scaler_std = X_train.std(axis=0)
X_train = (X_train - scaler_mean) / (scaler_std + 1e-5)
X_test = (X_test - scaler_mean) / (scaler_std + 1e-5)
t_end_feat = time.time() - t0_feat
print(f"Feature Engineering Time: {t_end_feat:.2f}s")

## 4. LightGBM Tabular Baseline

In [ ]:
print("\n--- LightGBM (with Weighted PageRank) ---")
t0_lgb = time.time()
clf = lgb.LGBMClassifier(n_estimators=100, random_state=42, n_jobs=-1, verbose=-1)
clf.fit(X_train, y_train)
preds_lgb = clf.predict_proba(X_test)[:, 1]
t_end_lgb = time.time() - t0_lgb

precision_lgb, recall_lgb, _ = precision_recall_curve(y_test, preds_lgb)
pr_auc_lgb = auc(recall_lgb, precision_lgb)
print(f'LightGBM PR-AUC: {pr_auc_lgb:.4f} (Model Time: {t_end_lgb:.2f}s)')

## 5. Pure PyTorch GraphSAGE Baseline

In [ ]:
print("\n--- Pure PyTorch GraphSAGE ---")
t0_gnn = time.time()
torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = PureTorchGraphSAGE(in_channels=X_train.shape[1], hidden_channels=32, out_channels=2).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

x_train_t = torch.tensor(X_train, dtype=torch.float32).to(device)
y_train_t = torch.tensor(y_train, dtype=torch.long).to(device)
ei_train_t = edge_index_train.to(device)
ew_train_t = ew_train.to(device)

x_test_t = torch.tensor(X_test, dtype=torch.float32).to(device)
ei_test_t = edge_index_test.to(device)
ew_test_t = ew_test.to(device)

for epoch in range(150):
    model.train()
    optimizer.zero_grad()
    out = model(x_train_t, ei_train_t, ew_train_t)
    loss = F.cross_entropy(out, y_train_t)
    loss.backward()
    optimizer.step()

model.eval()
with torch.no_grad():
    out = model(x_test_t, ei_test_t, ew_test_t)
    preds_gnn = F.softmax(out, dim=1)[:, 1].cpu().numpy()

t_end_gnn = time.time() - t0_gnn
precision_gnn, recall_gnn, _ = precision_recall_curve(y_test, preds_gnn)
pr_auc_gnn = auc(recall_gnn, precision_gnn)
print(f'GraphSAGE PR-AUC: {pr_auc_gnn:.4f} (Model Time: {t_end_gnn:.2f}s)')

## 6. Plotting the Results

In [ ]:
plt.figure(figsize=(8, 6))
plt.plot(recall_lgb, precision_lgb, label=f'LightGBM (PR-AUC = {pr_auc_lgb:.4f})', color='orange', linewidth=2)
plt.plot(recall_gnn, precision_gnn, label=f'GraphSAGE (PR-AUC = {pr_auc_gnn:.4f})', color='blue', linewidth=2)
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Inductive Precision-Recall Curve: LightGBM vs GraphSAGE')
plt.legend()
plt.grid(True)
plt.show()